# SHINRA Phase 0.1 — A100 bring-up

Не обучаем. Не `from_pretrained`. Random `ShinraForCausalLM` на A100 80GB.
RTX 2080 forbidden. Архитектуру не трогаем.

In [ ]:
import torch
import transformers
print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)
assert torch.cuda.is_available()
assert "2080" not in torch.cuda.get_device_name(0)
x = torch.randn(8, 8, device="cuda", dtype=torch.bfloat16)
assert (x @ x.T).dtype == torch.bfloat16

In [ ]:
import os
if not os.path.exists("/content/NULLXES-SHINRA-4B-INSTRUCT"):
    !git clone https://github.com/MagistrTheOne/NULLXES-SHINRA-4B-INSTRUCT.git /content/NULLXES-SHINRA-4B-INSTRUCT
%cd /content/NULLXES-SHINRA-4B-INSTRUCT
!pip install -q -e .
import os as _os
_os.environ["PYTHONPATH"] = "/content/NULLXES-SHINRA-4B-INSTRUCT"

In [ ]:
from model import ShinraConfig, ShinraForCausalLM
import torch

config = ShinraConfig.from_yaml("configs/pretrain_colab_100m.yaml")
model = ShinraForCausalLM(config)
n = sum(p.numel() for p in model.parameters())
print(n)
assert n == 3926076416
assert model.model.embed_tokens.weight.data_ptr() == model.lm_head.weight.data_ptr()

In [ ]:
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = model.to(dtype=torch.bfloat16, device="cuda")
print(next(model.parameters()).dtype)
assert next(model.parameters()).dtype == torch.bfloat16

In [ ]:
input_ids = torch.randint(0, 131072, (2, 2048), device="cuda")
model.train()
out = model(input_ids=input_ids, labels=input_ids, use_cache=False)
print(out.logits.shape)
assert tuple(out.logits.shape) == (2, 2048, 131072)
assert torch.isfinite(out.loss)

In [ ]:
out.loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
print(float(out.loss.detach()), float(grad_norm))
assert torch.isfinite(grad_norm)

In [ ]:
from training.optim import build_optimizer
opt = build_optimizer(model, 3e-4, 0.1, (0.9, 0.95), 1e-8)
opt.step()
opt.zero_grad(set_to_none=True)
print("optimizer_step_ok")

In [ ]:
from tokenizer.special_tokens import ALL_SPECIAL_TOKENS, EOT, END_OF_TEXT
print("n_specials", len(ALL_SPECIAL_TOKENS))
print("eot", EOT, "end_of_text", END_OF_TEXT)
assert "<|end|>" not in ALL_SPECIAL_TOKENS
assert "<|tool|>" not in ALL_SPECIAL_TOKENS

## Stop here

Phase 1 (tokenizer train / pack / 100M) — только после зелёного 0.1. Не запускать в этой сессии.